# A5. 库存补货决策 Notebook

> **配套模块**: [A5 库存与供应链](../paths/a-operators/a5-inventory.md)
>
> **功能**: 基于销售趋势预测补货量和时间 + 安全库存计算
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a5-inventory-reorder.ipynb)

---

## 1. 安装依赖

In [ ]:
!pip install -q pandas numpy plotly

## 2. 输入库存数据

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

# === 替换为你的真实数据 ===
skus = [
    {
        'sku': 'SKU-001', 'name': 'Wireless Earbuds',
        'current_inventory': 850,
        'daily_sales_avg': 15, 'daily_sales_std': 5,
        'lead_time_days': 45,  # 从下单到入仓的天数
        'moq': 500,            # 最小起订量
        'unit_cost': 8.50,
        'fba_monthly_storage': 0.15  # 月仓储费/件
    },
    {
        'sku': 'SKU-002', 'name': 'Phone Stand',
        'current_inventory': 1200,
        'daily_sales_avg': 27, 'daily_sales_std': 8,
        'lead_time_days': 35,
        'moq': 1000,
        'unit_cost': 3.20,
        'fba_monthly_storage': 0.10
    },
    {
        'sku': 'SKU-003', 'name': 'LED Desk Lamp',
        'current_inventory': 180,
        'daily_sales_avg': 10, 'daily_sales_std': 4,
        'lead_time_days': 50,
        'moq': 300,
        'unit_cost': 7.00,
        'fba_monthly_storage': 0.25
    }
]

SERVICE_LEVEL = 0.95  # 95% 服务水平（不缺货概率）
Z_SCORE = 1.65        # 对应 95% 服务水平的 Z 值

print(f'Loaded {len(skus)} SKUs')
for s in skus:
    days = s['current_inventory'] / s['daily_sales_avg']
    status = '🔴' if days < 14 else ('🟡' if days < 30 else '🟢')
    print(f"{status} {s['name']}: {s['current_inventory']} units, {days:.0f} days of supply")

## 3. 安全库存 + 补货点计算

In [ ]:
results = []
for s in skus:
    # 安全库存 = Z × σ_daily × √(lead_time)
    safety_stock = Z_SCORE * s['daily_sales_std'] * np.sqrt(s['lead_time_days'])
    
    # 补货点 = (日均销量 × 交货期) + 安全库存
    reorder_point = (s['daily_sales_avg'] * s['lead_time_days']) + safety_stock
    
    # 可售天数
    days_of_supply = s['current_inventory'] / s['daily_sales_avg']
    
    # 需要补货吗？
    needs_reorder = s['current_inventory'] <= reorder_point
    
    # 建议补货量（覆盖 60 天 + 安全库存）
    target_inventory = s['daily_sales_avg'] * 60 + safety_stock
    reorder_qty = max(target_inventory - s['current_inventory'] + s['daily_sales_avg'] * s['lead_time_days'], s['moq'])
    reorder_qty = int(np.ceil(reorder_qty / s['moq']) * s['moq'])  # 向上取整到 MOQ
    
    # 预计缺货日期
    stockout_date = datetime.now() + timedelta(days=days_of_supply)
    
    # 最晚下单日期
    latest_order_date = stockout_date - timedelta(days=s['lead_time_days'])
    
    # 补货成本
    reorder_cost = reorder_qty * s['unit_cost']
    
    results.append({
        'SKU': s['sku'], 'Product': s['name'],
        'Current': s['current_inventory'],
        'Daily Avg': s['daily_sales_avg'],
        'Days Supply': round(days_of_supply, 1),
        'Safety Stock': round(safety_stock),
        'Reorder Point': round(reorder_point),
        'Needs Reorder': needs_reorder,
        'Reorder Qty': reorder_qty,
        'Reorder Cost': reorder_cost,
        'Stockout Date': stockout_date.strftime('%Y-%m-%d'),
        'Latest Order': latest_order_date.strftime('%Y-%m-%d'),
        'Lead Time': s['lead_time_days']
    })

df = pd.DataFrame(results)
print('=== 补货决策分析 ===')
for _, r in df.iterrows():
    emoji = '🔴 立即补货' if r['Needs Reorder'] else '🟢 暂不需要'
    print(f"\n{emoji}: {r['Product']} ({r['SKU']})")
    print(f"  库存: {r['Current']} 件 | 可售: {r['Days Supply']} 天")
    print(f"  安全库存: {r['Safety Stock']} 件 | 补货点: {r['Reorder Point']} 件")
    if r['Needs Reorder']:
        print(f"  ⚡ 建议补货: {r['Reorder Qty']} 件 (${r['Reorder Cost']:,.0f})")
        print(f"  ⏰ 最晚下单: {r['Latest Order']} | 预计缺货: {r['Stockout Date']}")
    else:
        print(f"  预计缺货: {r['Stockout Date']}")

## 4. 库存消耗预测可视化

In [ ]:
fig = go.Figure()
colors = ['blue', 'green', 'orange']

for i, s in enumerate(skus):
    days_range = range(0, 90)
    inventory_projection = [max(s['current_inventory'] - s['daily_sales_avg'] * d, 0) for d in days_range]
    dates = [datetime.now() + timedelta(days=d) for d in days_range]
    
    fig.add_trace(go.Scatter(
        x=dates, y=inventory_projection,
        name=s['name'], line=dict(color=colors[i])
    ))
    
    # 补货点线
    rp = results[i]['Reorder Point']
    fig.add_hline(y=rp, line_dash='dash', line_color=colors[i],
                  annotation_text=f"{s['name']} Reorder Point")

fig.add_hline(y=0, line_color='red', line_width=2, annotation_text='STOCKOUT')
fig.update_layout(title='90-Day Inventory Projection', xaxis_title='Date', yaxis_title='Units')
fig.show()

## 5. 导出

In [ ]:
df.to_csv('reorder_decisions.csv', index=False)
print('Exported to reorder_decisions.csv')

# 紧急补货汇总
urgent = df[df['Needs Reorder']]
if len(urgent) > 0:
    total_cost = urgent['Reorder Cost'].sum()
    print(f'\n⚠️ {len(urgent)} SKUs need immediate reorder')
    print(f'Total reorder cost: ${total_cost:,.0f}')
else:
    print('\n✅ All SKUs have sufficient inventory')